# BindCraft — local control panel

Form fields for the settings, same idea as the Colab notebook's cell 3 and cell 9.
The difference: **this notebook does not run the campaign.** It writes the settings
file and hands the run to `run.sh`, which launches it in a `tmux` session on the
box. Closing this tab, or losing the SSH connection, does not stop the run.

Run the cells in order. Cell 1 is the form, cell 2 watches progress, cell 3 shows
results.

Open a terminal and `tmux attach -t bindcraft` any time you want the raw output.

In [ ]:
# 1. Campaign settings
import json, os, pathlib, subprocess
import ipywidgets as W
from IPython.display import display, clear_output

HERE  = pathlib.Path.cwd()
LOCAL = HERE if (HERE / "run.sh").exists() else HERE / "local"
REPO  = LOCAL.parent
BC_ROOT = pathlib.Path(os.environ.get("BC_ROOT", "/workspace"))
BC_DIR  = BC_ROOT / "bindcraft"
SETTINGS_DIR = LOCAL / "settings"

STATE = {"settings": None, "design_path": str(BC_ROOT / "designs")}

def _presets(sub):
    found = sorted(p.name for p in (BC_DIR / sub).glob("*.json"))
    return found or ["(none found)"]

def _expand(v):
    return str(v).replace("{{REPO}}", str(REPO)).replace("{{BC_ROOT}}", str(BC_ROOT))

row = dict(style={"description_width": "150px"}, layout=W.Layout(width="640px"))

_files = sorted(p.name for p in SETTINGS_DIR.glob("*.json"))
w_file     = W.Dropdown(options=_files + ["<new file>"],
                        value=_files[0] if _files else "<new file>",
                        description="Settings file", **row)
w_newname  = W.Text(placeholder="mytarget.json", description="Save as", **row)
w_name     = W.Text(description="Campaign name", **row)
w_pdb      = W.Text(description="Target PDB", **row)
w_chains   = W.Text(description="Chain(s)", **row)
w_hotspots = W.Text(description="Hotspot residues",
                    placeholder="A31,A34 — blank lets BindCraft choose", **row)
w_lengths  = W.IntRangeSlider(min=15, max=80, step=1, description="Binder length",
                              continuous_update=False, **row)
w_designs  = W.IntText(description="Final designs", **row)
w_path     = W.Text(description="Output folder", **row)

w_filters  = W.Dropdown(options=_presets("settings_filters"), description="Filters", **row)
w_advanced = W.Dropdown(options=_presets("settings_advanced"), description="Protocol", **row)
w_preset   = W.Dropdown(options=["Production defaults",
                                 "Fast debug — designs are meaningless",
                                 "Custom"],
                        description="Preset", **row)
w_plddt    = W.FloatSlider(value=0.65, min=0.10, max=0.90, step=0.05,
                           description="pLDDT gate", readout_format=".2f",
                           continuous_update=False, disabled=True, **row)

warn = W.HTML()
out  = W.Output()

def _check_lengths(*_):
    lo, _hi = w_lengths.value
    warn.value = ("<b style='color:#c0392b'>Binders under ~20 residues cannot pass "
                  "the quality gate. BindCraft will reject everything and loop "
                  "without accepting a design.</b>") if lo < 20 else ""

def _sync_preset(*_):
    fast = w_preset.value.startswith("Fast debug")
    w_plddt.disabled = w_preset.value != "Custom"
    if w_preset.value == "Production defaults":
        w_plddt.value = 0.65
    w_filters.disabled = w_advanced.disabled = fast
    warn.value = ("<i>Fast debug overrides the filters and protocol below, and "
                  "stops after one design.</i>") if fast else warn.value

def _load(*_):
    name = w_file.value
    if name == "<new file>":
        s = {"binder_name": "mytarget", "starting_pdb": "", "chains": "A",
             "target_hotspot_residues": "", "lengths": [25, 40],
             "number_of_final_designs": 10,
             "design_path": f"{BC_ROOT}/designs/mytarget/"}
        w_newname.value = "mytarget.json"
    else:
        s = json.load(open(SETTINGS_DIR / name))
        w_newname.value = name
    w_name.value     = s.get("binder_name", "")
    w_pdb.value      = _expand(s.get("starting_pdb", ""))
    w_chains.value   = s.get("chains", "A")
    w_hotspots.value = s.get("target_hotspot_residues") or ""
    lo, hi           = s.get("lengths", [25, 40])
    w_lengths.value  = (int(lo), int(hi))
    w_designs.value  = int(s.get("number_of_final_designs", 10))
    w_path.value     = _expand(s.get("design_path", f"{BC_ROOT}/designs/{w_name.value}/"))
    _check_lengths()

def _collect():
    lo, hi = w_lengths.value
    return {"design_path": w_path.value.rstrip("/") + "/",
            "binder_name": w_name.value.strip(),
            "starting_pdb": w_pdb.value.strip(),
            "chains": w_chains.value.strip(),
            "target_hotspot_residues": w_hotspots.value.strip(),
            "lengths": [int(lo), int(hi)],
            "number_of_final_designs": int(w_designs.value)}

def _write():
    name = (w_newname.value or w_file.value).strip()
    if not name.endswith(".json"):
        name += ".json"
    path = SETTINGS_DIR / name
    path.write_text(json.dumps(_collect(), indent=4) + "\n")
    if name not in w_file.options:
        w_file.options = sorted(p.name for p in SETTINGS_DIR.glob("*.json")) + ["<new file>"]
    w_file.value = name
    STATE["settings"] = str(path)
    STATE["design_path"] = _collect()["design_path"]
    return path

def _on_save(_):
    with out:
        clear_output()
        print("Saved", _write())

def _on_run(_):
    with out:
        clear_output()
        path = _write()
        cmd = ["bash", str(LOCAL / "run.sh"), "-s", str(path)]
        if not w_preset.value.startswith("Fast debug"):
            cmd += ["-f", str(BC_DIR / "settings_filters" / w_filters.value),
                    "-a", str(BC_DIR / "settings_advanced" / w_advanced.value)]
            if w_preset.value == "Custom":
                cmd += ["--plddt", f"{w_plddt.value:.2f}"]
        else:
            cmd.append("--smoke")
        print(" ".join(cmd), "\n")
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout, r.stderr, sep="")
        if r.returncode != 0:
            print(f"\nrun.sh exited {r.returncode} — nothing was launched.")

b_save = W.Button(description="Save settings", icon="save")
b_run  = W.Button(description="Save and run", button_style="success", icon="play")
b_save.on_click(_on_save)
b_run.on_click(_on_run)

w_file.observe(_load, names="value")
w_lengths.observe(_check_lengths, names="value")
w_preset.observe(_sync_preset, names="value")
_load()

display(W.VBox([
    W.HTML("<h4>Target</h4>"), w_file, w_newname, w_name, w_pdb, w_chains,
    w_hotspots, w_lengths, w_designs, w_path,
    W.HTML("<h4>Protocol</h4>"), w_preset, w_filters, w_advanced, w_plddt,
    warn, W.HBox([b_save, b_run]), out,
]))

In [ ]:
# 2. Watch the run
import pathlib, subprocess
import pandas as pd
import ipywidgets as W
from IPython.display import display, clear_output

w_out = W.Output()
b_ref = W.Button(description="Refresh", icon="refresh")

def _refresh(_=None):
    with w_out:
        clear_output()
        dp = pathlib.Path(STATE["design_path"])
        alive = subprocess.run(["tmux", "has-session", "-t", "bindcraft"],
                               capture_output=True).returncode == 0
        print("tmux session 'bindcraft':", "running" if alive else "not running")
        print("output folder:", dp)
        if not dp.is_dir():
            print("\nNothing there yet.")
            return
        for label, fn in (("Sampled trajectories", "trajectory_stats.csv"),
                          ("MPNN designs", "mpnn_design_stats.csv"),
                          ("Accepted designs", "final_design_stats.csv")):
            f = dp / fn
            n = len(pd.read_csv(f)) if f.exists() else 0
            print(f"{label:22s} {n}")
        logs = sorted(dp.glob("run_*.log"))
        if logs:
            print(f"\n--- last 25 lines of {logs[-1].name} ---")
            print(subprocess.run(["tail", "-n", "25", str(logs[-1])],
                                 capture_output=True, text=True).stdout)

b_ref.on_click(_refresh)
display(W.VBox([b_ref, w_out]))
_refresh()

In [ ]:
# 3. Results
import pathlib
import pandas as pd
from IPython.display import display

dp = pathlib.Path(STATE["design_path"])
final = dp / "final_design_stats.csv"

if not final.exists():
    print("No final_design_stats.csv yet — nothing has been accepted.")
else:
    df = pd.read_csv(final)
    if df.empty:
        print("No designs accepted yet.")
    else:
        cols = [c for c in ["Rank", "Design", "Sequence", "Length", "pLDDT", "i_pTM",
                            "pAE", "Binder_Energy_Score", "ShapeComplementarity",
                            "Interface_Hydrophobicity", "n_InterfaceResidues"]
                if c in df.columns]
        display(df[cols] if cols else df)

ranked = sorted((dp / "Accepted" / "Ranked").glob("*.pdb"))
if not ranked:
    print("\nNothing in Accepted/Ranked yet.")
else:
    print(f"\n{len(ranked)} ranked design(s). Showing {ranked[0].name}")
    try:
        import py3Dmol
        v = py3Dmol.view(width=700, height=500)
        v.addModel(ranked[0].read_text(), "pdb")
        v.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgrey"}})
        v.setStyle({"chain": "B"}, {"cartoon": {"color": "orange"}})
        v.zoomTo()
        v.show()
    except Exception as e:
        print("py3Dmol unavailable:", e)